# Snippet & Lexicon-Based Sentiment Analysis

This notebook implements a lexicon-based snippet approach for ESG sentiment analysis.
It generates rule-based sentiment scores using ESG-specific keywords, negation handling,
and intensifier detection. The outputs are saved for later hybrid fusion with transformer results.

In [9]:
# 1. Setup
import json
import pandas as pd

# Load lemmatized corpus (created in preprocessing)
with open("../data/cleaned_v2/preprocessed_with_lemma.jsonl", "r") as f:
    corpus = [json.loads(line) for line in f]

print(f"✅ Loaded lemmatized corpus | docs: {len(corpus)}")

✅ Loaded lemmatized corpus | docs: 9


In [6]:
# 2. Define ESG Lexicon
ESG_LEXICON = {
    "positive": [
        "sustainable", "renewable", "green", "inclusive", "responsible",
        "net", "zero", "diversity", "environmental", "governance", "social",
        "ethical", "recycling", "efficiency", "compliance", "innovation",
        "equity", "fairness", "biodiversity", "community", "wellbeing"
    ],
    "negative": [
        "emission", "emissions", "pollution", "scandal", "deforestation",
        "fine", "controversy", "risk", "hazard", "lawsuit", "waste",
        "shortage", "violation", "fraud", "breach", "exploitation",
        "child", "forced", "toxic", "unethical"
    ]
}

NEGATIONS = ["no", "not", "never", "none", "without"]
INTENSIFIERS = ["very", "highly", "extremely", "significantly"]

print("✅ Lexicon, negations, and intensifiers defined")

✅ Lexicon, negations, and intensifiers defined


In [10]:
# 3. Scoring Function
def score_tokens(tokens):
    pos, neg = 0, 0
    for i, token in enumerate(tokens):
        # Positive / negative hits
        if token in ESG_LEXICON["positive"]:
            multiplier = 2 if (i > 0 and tokens[i-1] in INTENSIFIERS) else 1
            if i > 0 and tokens[i-1] in NEGATIONS:
                neg += 1 * multiplier
            else:
                pos += 1 * multiplier
        elif token in ESG_LEXICON["negative"]:
            multiplier = 2 if (i > 0 and tokens[i-1] in INTENSIFIERS) else 1
            if i > 0 and tokens[i-1] in NEGATIONS:
                pos += 1 * multiplier
            else:
                neg += 1 * multiplier
    return {"pos": pos, "neg": neg, "score": pos - neg}

In [11]:
# 4. Apply scoring to all documents
results = []
for doc in corpus:
    company = doc["company"]
    year = doc["year"]
    tokens = doc["tokens"]

    scores = score_tokens(tokens)
    results.append({
        "company": company,
        "year": year,
        "pos": scores["pos"],
        "neg": scores["neg"],
        "score": scores["score"]
    })

print(f"✅ Scored {len(results)} documents")

✅ Scored 9 documents


In [12]:
# 5. Save results
df = pd.DataFrame(results)
out_path = "../data/processed/snippet_scores.csv"
df.to_csv(out_path, index=False)

print(f"✅ Saved snippet lexicon scores to {out_path} | rows: {len(df)}")
print(df.head())

✅ Saved snippet lexicon scores to ../data/processed/snippet_scores.csv | rows: 9
  company  year  pos  neg  score
0  Google  2022  140  107     33
1  Google  2023  951  654    297
2  Google  2024  689  705    -16
3    HSBC  2022  818  672    146
4    HSBC  2023  829  785     44
